# Section 07d Onboarding Notebook: Bloch–Redfield Theory for Vibrational Energy Relaxation (VER) in Excited Electronic States

### Overview & Pedagogical Design
When an ultrashort laser pulse photoexcites a chromophore from its electronic ground state $|g\rangle$ ($S_0$) to an excited electronic state $|e\rangle$ ($S_1$), the sudden change in equilibrium geometry creates a non-stationary **vibrational wavepacket** high up on the $S_1$ potential energy surface (a superposition of Franck–Condon vibrational levels $|e, v=0, 1, 2, \dots\rangle$). In condensed-phase solution or protein environments, bilinear coupling between the chromophore's intramolecular vibrations and the surrounding thermal solvent bath dissipates this excess vibrational energy on picosecond timescales — a process known as **Vibrational Energy Relaxation (VER)**, which drives **Kasha's Rule** (relaxation to the $v'=0$ vibrational level of $S_1$ prior to fluorescence or internal conversion).

When the system–bath coupling $\lambda_{\text{bath}}$ is weak compared to the intramolecular vibrational spacing $\hbar\omega_{\text{vib}}$ and the bath correlation time $\tau_c = \gamma_c^{-1}$ is short (the **Born–Markov approximation**), the reduced density matrix $\hat{\rho}(t)$ in the **system eigenbasis** $\{|\alpha\rangle\}$ ($\hat{H}_S|\alpha\rangle = E_\alpha|\alpha\rangle$, $\omega_{\alpha\beta} = E_\alpha - E_\beta$) obeys the microscopic **[Bloch–Redfield Master Equation (BRME)](https://qutip.org/docs/4.7/guide/guide-heom.html)** ([Janković PRB 2022](https://www.scl.rs/papers/Jankovic_PRB_2022.pdf); [Lindbladian & Redfield Formalism](https://en.wikipedia.org/wiki/Lindbladian)):

$$\frac{d\rho_{ab}(t)}{dt} = -i\omega_{ab}\,\rho_{ab}(t) + \sum_{c,d} R_{abcd}\,\rho_{cd}(t)$$

Unlike phenomenological Lindblad equations with hand-picked jump operators, **Redfield theory** derives the relaxation rates directly from the **bath spectral density $J(\omega)$** and the **temperature $T$**, automatically enforcing **Quantum Detailed Balance** ($k_{a \to b} / k_{b \to a} = e^{-(E_b - E_a)/k_B T}$) so that the excited state thermalizes to the exact canonical Boltzmann distribution!

In this notebook, **no Redfield or vibronic functions are pre-solved for you**. You will implement **15 granular `# TODO` functions** covering the entire workflow:
- **Part 1 — Displaced Excited-State Vibronic Hamiltonian, Eigenbasis Transformation & Franck–Condon Photoexcitation (`TODO 1` – `TODO 4`)**
- **Part 2 — Bath Spectral Density $J(\omega)$, Bose–Einstein Thermal Power Spectrum & Detailed Balance (`TODO 5` – `TODO 7`)**
- **Part 3 — Constructing the 4-Index Redfield Tensor $R_{abcd}$, Secular Approximation & Pauli Golden-Rule Rates (`TODO 8` – `TODO 11`)**
- **Part 4 — Propagating Excited-State Vibrational Cooling, $T_1$ Lifetime Extraction & Secular vs. Non-Secular Dynamics (`TODO 12` – `TODO 15`)**
- **Part 5 — Step-by-Step Checkpoint & Verification Suite (`run_section07c_redfield_checkpoints`)**

In [ ]:
import numpy as np
from scipy.linalg import eigh, expm
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True, linewidth=110)
print("Section 07c Bloch-Redfield Vibrational Relaxation environment initialized. Implement TODOs 1-15 and run `run_section07c_redfield_checkpoints()`!")

## Part 1: Displaced Excited-State Vibronic Hamiltonian & Franck–Condon Photoexcitation (TODOs 1–4)

### Mathematical Specifications
We consider a two-electronic-state chromophore ($\{|g\rangle, |e\rangle\}$) coupled to a primary intramolecular vibrational mode with annihilation operator $\hat{b}$ of cutoff dimension $N_{\text{vib}}$ (total Hilbert space dimension $D = 2 N_{\text{vib}}$, setting $\hbar = 1$):

1. **Vibronic System Hamiltonian (`build_vibronic_system_hamiltonian` — TODO 1)**:
   - On the ground electronic state $|g\rangle\langle g|$ (block `0 .. n_vib-1`):
     $$\hat{H}_g = \omega_{\text{vib}}\,\hat{b}^\dagger\hat{b}$$
   - On the excited electronic state $|e\rangle\langle e|$ (block `n_vib .. 2*n_vib-1`), displaced by dimensionless Huang–Rhys parameter $\sqrt{S_{\text{HR}}}$ with small Morse-like anharmonicity $\chi_{\text{anh}}$:
     $$\hat{b}_e = \hat{b} - \sqrt{S_{\text{HR}}}\,\hat{I}_{\text{vib}}, \qquad \hat{N}_{\text{vib}}^{(e)} = \hat{b}_e^\dagger \hat{b}_e, \qquad \hat{H}_e = \omega_{eg}\,\hat{I}_{\text{vib}} + \omega_{\text{vib}}\,\hat{N}_{\text{vib}}^{(e)} - \chi_{\text{anh}}\left(\hat{N}_{\text{vib}}^{(e)}\right)^2$$
   - Diabatic internal-conversion coupling between $|g\rangle$ and $|e\rangle$: $V_{\text{ic}}\left(|g\rangle\langle e| + |e\rangle\langle g|\right) \otimes \hat{I}_{\text{vib}}$.
   - System–bath coupling operator (the intramolecular vibrational coordinate coupled to the solvent bath, plus weak anharmonic overtone coupling $\eta_{\text{over}}(\hat{b} + \hat{b}^\dagger)^2$):
     $$\hat{S} = \hat{I}_{\text{el}} \otimes \left[\left(\hat{b} + \hat{b}^\dagger\right) + \eta_{\text{over}}\left(\hat{b} + \hat{b}^\dagger\right)^2\right]$$

2. **Eigenbasis Transformation & Bohr Frequencies (`diagonalize_system_and_transform_coupling` — TODO 2)**:
   Diagonalize $\hat{H}_S = \mathbf{U}\,\text{diag}(\mathbf{E})\,\mathbf{U}^\dagger$ (`scipy.linalg.eigh`), form the Bohr frequency matrix $\omega_{\alpha\beta} = E_\alpha - E_\beta$ (`E[:, None] - E[None, :]`), and transform $\hat{S}$ into the eigenbasis:
   $$\mathbf{S}_{\text{eig}} = \mathbf{U}^\dagger\,\hat{S}\,\mathbf{U}$$

3. **Vertical Franck–Condon Photoexcitation (`prepare_franck_condon_vertical_state` — TODO 3)**:
   Prior to laser excitation, the molecule is in its ground electronic and vibrational vacuum state $|\psi_{\text{init}}\rangle = |g\rangle \otimes |0\rangle_{\text{vib}}$ (index `0`). An ultrashort delta-pulse vertically promotes this wavepacket to the unshifted $v=0$ state on the excited electronic surface:
   $$|\psi_{\text{FC}}(0)\rangle = |e\rangle \otimes |0\rangle_{\text{vib}} \quad (\text{site basis index } N_{\text{vib}})$$
   Construct $\hat{\rho}_{\text{site}}(0) = |\psi_{\text{FC}}(0)\rangle\langle\psi_{\text{FC}}(0)|$ and transform it to the system eigenbasis: $\boldsymbol{\rho}_{\text{eig}}(0) = \mathbf{U}^\dagger \hat{\rho}_{\text{site}}(0) \mathbf{U}$.

4. **Franck–Condon Vibronic Stick Spectrum (`compute_franck_condon_stick_spectrum` — TODO 4)**:
   For each system eigenstate $|\alpha\rangle$ in the excited manifold, compute its transition frequency $\Delta E_\alpha = E_\alpha - E_0$ and Franck–Condon intensity $I_\alpha = |\langle \alpha | \psi_{\text{FC}}(0) \rangle|^2 = |U_{N_{\text{vib}}, \alpha}|^2$.

In [ ]:
def build_vibronic_system_hamiltonian(
    n_vib: int = 6,
    omega_eg: float = 8.0,
    omega_vib: float = 1.0,
    huang_rhys_s: float = 1.2,
    chi_anh: float = 0.02,
    v_ic: float = 0.0,
    eta_over: float = 0.05,
) -> dict[str, np.ndarray]:
    '''
    TODO 1: Construct the 2*n_vib x 2*n_vib vibronic Hamiltonian `H_s`, system-bath coupling
    operator `S_op`, laboratory vibrational coordinate `Q_vib = I_el (x) (b + b_dag)`,
    excited-state displaced vibrational number operator `N_vib_e = |e><e| (x) (b_e^dag @ b_e)`,
    and excited electronic projector `P_e = |e><e| (x) I_vib`.
    Return a dict with keys 'H_s', 'S_op', 'Q_vib', 'N_vib_e', 'P_e', 'b', 'b_dag'.
    '''
    # TODO 1: Build single-mode b, b_dag of size n_vib, b_e = b - sqrt(huang_rhys_s)*I_vib,
    #         n_e = b_e.T @ b_e, H_g, H_e, and lift to 2*n_vib x 2*n_vib matrices.
    raise NotImplementedError("Complete TODO 1: build_vibronic_system_hamiltonian")


def diagonalize_system_and_transform_coupling(
    H_s: np.ndarray, S_op: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    '''
    TODO 2: Diagonalize H_s with scipy.linalg.eigh -> (E_evals, U_eigvecs), construct the
    Bohr frequency matrix omega_mat[a, b] = E_evals[a] - E_evals[b], and transform S_op
    into the eigenbasis S_eig = U_eigvecs.T @ S_op @ U_eigvecs.
    Return (E_evals, U_eigvecs, omega_mat, S_eig).
    '''
    # TODO 2: Diagonalize H_s, form omega_mat via broadcasting, and transform S_eig.
    raise NotImplementedError("Complete TODO 2: diagonalize_system_and_transform_coupling")


def prepare_franck_condon_vertical_state(
    U_eigvecs: np.ndarray, n_vib: int
) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 3: Construct the initial vertical Franck-Condon density matrix in the site basis
    `rho_site_0 = |e, 0><e, 0|` (where |e, 0> is basis index `n_vib`) and in the system
    eigenbasis `rho_eig_0 = U_eigvecs.T @ rho_site_0 @ U_eigvecs` (complex128).
    Return (rho_site_0, rho_eig_0).
    '''
    # TODO 3: Create psi_fc of length 2*n_vib with 1.0 at index n_vib, form rho_site_0, and transform to rho_eig_0.
    raise NotImplementedError("Complete TODO 3: prepare_franck_condon_vertical_state")


def compute_franck_condon_stick_spectrum(
    E_evals: np.ndarray, U_eigvecs: np.ndarray, n_vib: int
) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 4: Compute excitation frequencies `delta_e = E_evals - E_evals[0]` and Franck-Condon
    overlap weights `fc_weights = |U_eigvecs[n_vib, :]|^2`.
    Return (delta_e, fc_weights).
    '''
    # TODO 4: Compute delta_e and fc_weights and verify that sum(fc_weights) == 1.0.
    raise NotImplementedError("Complete TODO 4: compute_franck_condon_stick_spectrum")

## Part 2: Bath Spectral Density $J(\omega)$, Thermal Power Spectrum & Detailed Balance (TODOs 5–7)

### Mathematical Specifications
1. **Ohmic–Lorentzian (Drude–Lorentz) Bath Spectral Density (`evaluate_spectral_density` — TODO 5)**:
   For frequency $\omega > 0$, bath reorganization energy $\lambda_{\text{bath}}$, and cutoff frequency $\gamma_c$:
   $$J(\omega) = 2\lambda_{\text{bath}}\,\frac{\gamma_c\,\omega}{\omega^2 + \gamma_c^2}$$

2. **Real Half-Fourier Bath Correlation Matrix $W(\omega_{\alpha\beta})$ (`compute_bath_rate_matrix_W` — TODO 6)**:
   For any transition frequency $\omega = \omega_{\alpha\beta} = E_\alpha - E_\beta$ at thermal energy $k_B T$:
   $$W(\omega) = \begin{cases} \pi\,J(\omega)\,\big[n_{\text{BE}}(\omega, T) + 1\big] & \omega > 10^{-12} \quad (\text{Spontaneous + stimulated emission into bath}) \\ \pi\,J(|\omega|)\,n_{\text{BE}}(|\omega|, T) & \omega < -10^{-12} \quad (\text{Thermal absorption from bath}) \\ \frac{2\pi\,\lambda_{\text{bath}}\,k_B T}{\gamma_c} & |\omega| \le 10^{-12} \quad (\text{Pure dephasing limit } \omega \to 0) \end{cases}$$
   where $n_{\text{BE}}(\Omega, T) = \frac{1}{e^{\Omega / k_B T} - 1}$ (and $n_{\text{BE}} = 0$ as $k_B T \to 0$).

3. **Quantum Detailed Balance Verification (`verify_detailed_balance_kms` — TODO 7)**:
   Verify across all pairs $(\alpha, \beta)$ with $\omega_{\alpha\beta} > 10^{-8}$ that:
   $$\frac{W(\omega_{\beta\alpha})}{W(\omega_{\alpha\beta})} = \frac{W(-\omega_{\alpha\beta})}{W(+\omega_{\alpha\beta})} = \exp\!\left(-\frac{\omega_{\alpha\beta}}{k_B T}\right)$$
   This KMS condition guarantees that any non-degenerate system coupled to the bath relaxes to the canonical Gibbs state $\rho_{\alpha\alpha}^{\text{eq}} \propto e^{-E_\alpha / k_B T}$!

In [ ]:
def evaluate_spectral_density(
    omega: float | np.ndarray, lambda_bath: float = 0.015, gamma_cutoff: float = 1.5
) -> float | np.ndarray:
    '''
    TODO 5: Evaluate the Ohmic-Lorentzian spectral density J(omega) = 2 * lambda_bath * gamma_cutoff * omega / (omega^2 + gamma_cutoff^2)
    for positive frequency `omega`.
    '''
    # TODO 5: Implement the Ohmic-Lorentzian formula.
    raise NotImplementedError("Complete TODO 5: evaluate_spectral_density")


def compute_bath_rate_matrix_W(
    omega_mat: np.ndarray,
    lambda_bath: float = 0.015,
    gamma_cutoff: float = 1.5,
    temperature_kt: float = 0.25,
) -> np.ndarray:
    '''
    TODO 6: Evaluate the (D, D) bath correlation rate matrix W_mat[a, b] = W(omega_mat[a, b])
    handling omega > 1e-12 (emission), omega < -1e-12 (absorption), and |omega| <= 1e-12 (pure dephasing).
    '''
    # TODO 6: Allocate W_mat of shape omega_mat.shape and populate the three frequency cases.
    raise NotImplementedError("Complete TODO 6: compute_bath_rate_matrix_W")


def verify_detailed_balance_kms(
    W_mat: np.ndarray, omega_mat: np.ndarray, temperature_kt: float
) -> float:
    '''
    TODO 7: Return the maximum absolute error max_{omega_{ab} > 1e-8} |W_{ba} / W_{ab} - exp(-omega_{ab} / kT)|.
    '''
    # TODO 7: Mask pairs (a, b) with omega_mat[a, b] > 1e-8 and compute max |W_mat.T / W_mat - exp(-omega_mat / kT)|.
    raise NotImplementedError("Complete TODO 7: verify_detailed_balance_kms")

## Part 3: Constructing the 4-Index Redfield Tensor $R_{abcd}$, Secular Approximation & Pauli Rates (TODOs 8–11)

### Mathematical Specifications
1. **Four-Index Bloch–Redfield Relaxation Tensor (`build_redfield_relaxation_tensor` — TODO 8)**:
   In the system eigenbasis $\{|a\rangle\}$, the dissipative part of $\dot{\rho}_{ab} = -i\omega_{ab}\rho_{ab} + \sum_{cd} R_{abcd}\rho_{cd}$ is governed by the 4-index **Redfield tensor** $R_{abcd}$ ([QuTiP Bloch–Redfield Guide](https://qutip.org/docs/4.7/guide/guide-heom.html)):
   $$R_{abcd} = S_{ac}\,S_{db}\,\big[W_{ca} + W_{db}\big] - \delta_{bd}\sum_k S_{ak}\,S_{kc}\,W_{ck} - \delta_{ac}\sum_k S_{dk}\,S_{kb}\,W_{dk}$$
   where $W_{ca} = W(\omega_{ca}) = W(E_c - E_a)$.
   *Fundamental Trace Identity:* Summing over $a = b$ (`np.einsum("aacd->cd", R_tensor)`) must vanish identically ($\sum_a R_{aacd} = 0\ \forall c,d$) so that $\frac{d}{dt}\text{Tr}[\hat{\rho}(t)] = 0$!

2. **Secular (Rotating-Wave / Davies) Approximation (`apply_secular_approximation` — TODO 9)**:
   In the interaction picture, each term $R_{abcd}$ oscillates at frequency $\omega_{ab} - \omega_{cd}$. When the relaxation rates are much smaller than the Bohr frequency splittings, rapidly oscillating non-resonant terms average to zero:
   $$R_{abcd}^{\text{sec}} = \begin{cases} R_{abcd} & |\omega_{ab} - \omega_{cd}| < \epsilon_{\text{sec}} \\ 0 & |\omega_{ab} - \omega_{cd}| \ge \epsilon_{\text{sec}} \end{cases}$$

3. **Pauli Master Equation Transition Matrix & Coherence Dephasing Rates (`extract_pauli_rates_and_dephasing` — TODO 10)**:
   From $R_{abcd}$:
   - The diagonal population transfer matrix is $K_{ac} = R_{aacc}$, where for $a \ne c$, $K_{ac} = 2 W_{ca} |S_{ac}|^2 \ge 0$ is the **Fermi Golden Rule transition rate** from eigenstate $|c\rangle \to |a\rangle$, and $K_{cc} = -\sum_{a \ne c} K_{ac}$ is the total depopulation rate out of $|c\rangle$.
   - The coherence dephasing rate of $\rho_{ab}$ ($a \ne b$) is $\gamma_{ab} = -R_{abab} = \frac{1}{2}(|K_{aa}| + |K_{bb}|) + \gamma_{ab}^{\text{pure}}$.

4. **Superoperator Matrix Representation (`build_redfield_superoperator` — TODO 11)**:
   Reshaping $(a, b) \mapsto I = a D + b$ and $(c, d) \mapsto J = c D + d$ converts the Bloch–Redfield master equation into a $D^2 \times D^2$ linear superoperator ODE $\frac{d}{dt}|\hat{\rho}\rangle\!\rangle = \mathcal{L}_{\text{BR}}|\hat{\rho}\rangle\!\rangle$:
   $$(\mathcal{L}_{\text{BR}})_{ab, cd} = -i\omega_{ab}\,\delta_{ac}\delta_{bd} + R_{abcd}$$

In [ ]:
def build_redfield_relaxation_tensor(
    S_eig: np.ndarray, W_mat: np.ndarray
) -> np.ndarray:
    '''
    TODO 8: Construct the 4-index Bloch-Redfield relaxation tensor R[a, b, c, d] of shape (D, D, D, D)
    from real symmetric eigenbasis coupling matrix `S_eig` and rate matrix `W_mat` (where W_mat[c, a] = W(E_c - E_a)):
    R_{abcd} = S_{ac} S_{db} (W_{ca} + W_{db})
               - delta_{bd} * sum_k S_{ak} S_{kc} W_{ck}
               - delta_{ac} * sum_k S_{dk} S_{kb} W_{dk}.
    '''
    # TODO 8: Compute term1 = einsum("ac,db,ca->abcd", S_eig, S_eig, W_mat) + einsum("ac,db,db->abcd", S_eig, S_eig, W_mat),
    #          build the 2-index damping matrix M_{ac} = sum_k S_{ak} S_{kc} W_{ck}, and subtract
    #         einsum("ac,bd->abcd", M, I) + einsum("db,ac->abcd", M, I).
    raise NotImplementedError("Complete TODO 8: build_redfield_relaxation_tensor")


def apply_secular_approximation(
    R_tensor: np.ndarray, omega_mat: np.ndarray, sec_tol: float = 1e-6
) -> np.ndarray:
    '''
    TODO 9: Return a copy of `R_tensor` in which all elements with |omega_mat[a, b] - omega_mat[c, d]| >= sec_tol
    are set to zero.
    '''
    # TODO 9: Build mask = np.abs(omega_mat[:, :, None, None] - omega_mat[None, None, :, :]) < sec_tol and return R_tensor * mask.
    raise NotImplementedError("Complete TODO 9: apply_secular_approximation")


def extract_pauli_rates_and_dephasing(
    R_tensor: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 10: Extract the (D, D) Pauli population rate matrix K_mat[a, c] = R_tensor[a, a, c, c]
    and the (D, D) coherence dephasing rate matrix gamma_mat[a, b] = -R_tensor[a, b, a, b] (with zeros on diagonal).
    Return (K_mat, gamma_mat).
    '''
    # TODO 10: Extract K_mat = einsum("aacc->ac", R_tensor) and gamma_mat = -einsum("abab->ab", R_tensor) * (1 - I).
    raise NotImplementedError("Complete TODO 10: extract_pauli_rates_and_dephasing")


def build_redfield_superoperator(
    omega_mat: np.ndarray, R_tensor: np.ndarray
) -> np.ndarray:
    '''
    TODO 11: Construct the (D^2, D^2) Bloch-Redfield Liouvillian superoperator matrix L_br
    from `omega_mat` (D, D) and `R_tensor` (D, D, D, D) in row-major index order.
    '''
    # TODO 11: Reshape R_tensor to (D^2, D^2) and subtract 1j * diag(omega_mat.ravel()).
    raise NotImplementedError("Complete TODO 11: build_redfield_superoperator")

## Part 4: Propagating Excited-State Vibrational Cooling, $T_1$ Lifetimes & Thermalization (TODOs 12–15)

### Mathematical Specifications
1. **RK4 Step for the Bloch–Redfield Equation (`rk4_step_redfield` — TODO 12)**:
   In the system eigenbasis, the right-hand side is:
   $$\mathcal{F}(\boldsymbol{\rho}) = -i\boldsymbol{\omega} \odot \boldsymbol{\rho} + \text{\texttt{np.einsum("abcd,cd->ab", R\_tensor, rho)}}$$
   Advance $\boldsymbol{\rho}(t) \to \boldsymbol{\rho}(t + \Delta t)$ using 4th-order Runge–Kutta.

2. **Propagating Excited-State Vibrational Energy Relaxation (`propagate_redfield_ver_trajectory` — TODO 13)**:
   At each step $t_n$:
   - Transform $\boldsymbol{\rho}_{\text{eig}}(t_n)$ back to the site basis: $\boldsymbol{\rho}_{\text{site}}(t_n) = \mathbf{U}\,\boldsymbol{\rho}_{\text{eig}}(t_n)\,\mathbf{U}^\dagger$.
   - Record the **mean displaced excited-state vibrational quantum number** $\langle \hat{N}_{\text{vib}}^{(e)}(t_n) \rangle = \text{Re}\,\text{Tr}[\boldsymbol{\rho}_{\text{site}}(t_n)\,\hat{N}_{\text{vib}}^{(e)}]$, **coherent vibrational displacement** $\langle \hat{Q}_{\text{vib}}(t_n)\rangle = \text{Re}\,\text{Tr}[\boldsymbol{\rho}_{\text{site}}(t_n)\,\hat{Q}_{\text{vib}}]$, **eigenstate populations** $P_\alpha(t_n) = \text{Re}\,\rho_{\text{eig}, \alpha\alpha}(t_n)$, **trace error** $|\text{Tr}\,\boldsymbol{\rho} - 1|$, and **minimum eigenvalue** $\lambda_{\min}(\boldsymbol{\rho})$.

3. **Extracting the Vibrational Cooling ($T_1$) Lifetime (`fit_vibrational_relaxation_lifetime` — TODO 14)**:
   Fit the decay of the mean vibrational excitation $\langle \hat{N}_{\text{vib}}^{(e)}(t)\rangle - \langle \hat{N}_{\text{vib}}^{(e)}(\infty)\rangle \approx \Delta N_0\,e^{-t / \tau_{\text{VER}}}$ via linear regression on $\ln\big(\langle \hat{N}_{\text{vib}}^{(e)}(t)\rangle - \langle \hat{N}_{\text{vib}}^{(e)}(\infty)\rangle\big)$ vs. $t$ to extract the **vibrational energy relaxation time $\tau_{\text{VER}}$** ($T_1$).

4. **Boltzmann Thermalization & Secular vs. Full Redfield Comparison (`verify_excited_state_boltzmann_equilibrium` — TODO 15)**:
   When internal conversion is absent ($V_{\text{ic}} = 0$), the excited electronic manifold $\{|\alpha\rangle\}_{\alpha = N_{\text{vib}}}^{2N_{\text{vib}}-1}$ is an invariant subspace that relaxes toward the **intra-manifold canonical Boltzmann distribution** (**Kasha's Rule**):
   $$P_\alpha^{\text{Boltzmann}}(T) = \frac{e^{-(E_\alpha - E_{N_{\text{vib}}}) / k_B T}}{\sum_{\beta = N_{\text{vib}}}^{2N_{\text{vib}}-1} e^{-(E_\beta - E_{N_{\text{vib}}}) / k_B T}}$$

In [ ]:
def rk4_step_redfield(
    rho_eig: np.ndarray, omega_mat: np.ndarray, R_tensor: np.ndarray, dt: float
) -> np.ndarray:
    '''
    TODO 12: Advance the (D, D) complex128 density matrix `rho_eig` by one step `dt`
    using 4th-order Runge-Kutta on d(rho)/dt = -1j * omega_mat * rho + einsum("abcd,cd->ab", R_tensor, rho).
    '''
    # TODO 12: Define rhs(r) and evaluate k1, k2, k3, k4 to return updated rho_eig.
    raise NotImplementedError("Complete TODO 12: rk4_step_redfield")


def propagate_redfield_ver_trajectory(
    rho_eig_0: np.ndarray,
    U_eigvecs: np.ndarray,
    omega_mat: np.ndarray,
    R_tensor: np.ndarray,
    Q_vib_site: np.ndarray,
    N_vib_e_site: np.ndarray,
    dt: float = 0.25,
    n_steps: int = 320,
) -> dict[str, np.ndarray]:
    '''
    TODO 13: Propagate `rho_eig_0` for `n_steps` steps of size `dt` using `rk4_step_redfield`.
    Return a dict with keys:
    'times', 'mean_n_vib_e', 'mean_q_vib', 'eigen_populations' (shape n_steps+1, D),
    'trace_error', 'min_eigenvalue', 'final_rho_eig'.
    '''
    # TODO 13: Loop over steps, transforming rho_site = U @ rho_eig @ U.T to measure <N_vib_e> and <Q_vib>.
    raise NotImplementedError("Complete TODO 13: propagate_redfield_ver_trajectory")


def fit_vibrational_relaxation_lifetime(
    times: np.ndarray, mean_n_vib_e: np.ndarray, n_eq: float
) -> tuple[float, float]:
    '''
    TODO 14: Estimate the vibrational energy relaxation time constant `tau_ver` (T1) and decay rate
    `k_ver = 1 / tau_ver` by performing a linear least-squares fit of `log(mean_n_vib_e - n_eq)` vs `times`
    over all points where `mean_n_vib_e - n_eq > 0.05 * (mean_n_vib_e[0] - n_eq)`.
    Return (tau_ver, k_ver).
    '''
    # TODO 14: Filter valid points, fit slope via np.polyfit(t_valid, np.log(delta_n_valid), 1), and return (-1/slope, -slope).
    raise NotImplementedError("Complete TODO 14: fit_vibrational_relaxation_lifetime")


def verify_excited_state_boltzmann_equilibrium(
    final_eigen_pops: np.ndarray,
    E_evals: np.ndarray,
    n_vib: int,
    temperature_kt: float,
) -> tuple[np.ndarray, float]:
    '''
    TODO 15: Compute the analytical canonical Boltzmann distribution `boltzmann_pops` of length 2*n_vib
    restricted to the excited electronic manifold (indices n_vib .. 2*n_vib - 1, with 0 on indices 0 .. n_vib - 1),
    and return (boltzmann_pops, max_abs_error) where max_abs_error = max(|final_eigen_pops - boltzmann_pops|).
    '''
    # TODO 15: Compute unnormalized weights exp(-(E_evals[n_vib:] - E_evals[n_vib]) / temperature_kt),
    #          normalize over the excited manifold, and compare with final_eigen_pops.
    raise NotImplementedError("Complete TODO 15: verify_excited_state_boltzmann_equilibrium")

## Part 5: Step-by-Step Checkpoint & Verification Suite

Run `run_section07c_redfield_checkpoints()` below as you implement each `# TODO` function to test your displaced vibronic Hamiltonian, Franck–Condon wavepacket preparation, Ohmic–Lorentzian bath spectral density, KMS detailed balance, 4-index Bloch–Redfield tensor $R_{abcd}$, secular approximation, Pauli Golden-Rule rates, vibrational cooling trajectory, $T_1$ lifetime extraction, and excited-state Boltzmann thermalization!

In [2]:
def run_section07c_redfield_checkpoints() -> None:
    '''Step-by-step checkpoint verifier for all 15 #TODO functions in Section 07c (Redfield Theory & VER).'''
    passed = 0
    total_todos = 15

    try:
        # --- PART 1 CHECKS: VIBRONIC HAMILTONIAN & FRANCK-CONDON STATE (TODOs 1-4) ---
        n_v = 6
        sys_dict = build_vibronic_system_hamiltonian(n_vib=n_v, omega_eg=8.0, omega_vib=1.0, huang_rhys_s=1.2, chi_anh=0.02, v_ic=0.0)
        H_s, S_op, Q_vib, N_vib_e = sys_dict["H_s"], sys_dict["S_op"], sys_dict["Q_vib"], sys_dict["N_vib_e"]
        assert H_s.shape == (2 * n_v, 2 * n_v) and np.allclose(H_s, H_s.T)
        passed += 1
        print("[PASS] TODO 1 : build_vibronic_system_hamiltonian")

        E_evals, U_eig, omega_mat, S_eig = diagonalize_system_and_transform_coupling(H_s, S_op)
        assert np.allclose(omega_mat + omega_mat.T, 0.0) and np.allclose(U_eig.T @ U_eig, np.eye(2 * n_v))
        passed += 1
        print("[PASS] TODO 2 : diagonalize_system_and_transform_coupling")

        rho_site_0, rho_eig_0 = prepare_franck_condon_vertical_state(U_eig, n_vib=n_v)
        n_vib_init = float(np.trace(rho_site_0 @ N_vib_e).real)
        assert np.isclose(np.trace(rho_eig_0).real, 1.0) and np.isclose(n_vib_init, 1.2, atol=1e-10)
        passed += 1
        print(f"[PASS] TODO 3 : prepare_franck_condon_vertical_state (Initial Franck-Condon <N_vib^e(0)> = {n_vib_init:.4f} == S_HR)")

        delta_e, fc_w = compute_franck_condon_stick_spectrum(E_evals, U_eig, n_vib=n_v)
        assert np.isclose(np.sum(fc_w), 1.0, atol=1e-12) and np.sum(fc_w[:n_v]) < 1e-12
        passed += 1
        print("[PASS] TODO 4 : compute_franck_condon_stick_spectrum")

        # --- PART 2 CHECKS: SPECTRAL DENSITY & DETAILED BALANCE (TODOs 5-7) ---
        j_val = evaluate_spectral_density(1.5, lambda_bath=0.02, gamma_cutoff=1.5)
        assert np.isclose(j_val, 0.02)
        passed += 1
        print(f"[PASS] TODO 5 : evaluate_spectral_density (Peak J(gamma_c) = {j_val:.4f} == lambda_bath)")

        kT_val = 0.30
        W_mat = compute_bath_rate_matrix_W(omega_mat, lambda_bath=0.02, gamma_cutoff=1.5, temperature_kt=kT_val)
        assert W_mat.shape == (2 * n_v, 2 * n_v) and np.all(W_mat >= 0.0)
        passed += 1
        print("[PASS] TODO 6 : compute_bath_rate_matrix_W")

        kms_err = verify_detailed_balance_kms(W_mat, omega_mat, temperature_kt=kT_val)
        assert kms_err < 1e-12
        passed += 1
        print(f"[PASS] TODO 7 : verify_detailed_balance_kms (Max KMS detailed-balance error = {kms_err:.2e})")

        # --- PART 3 CHECKS: REDFIELD TENSOR, SECULAR APPROX & PAULI RATES (TODOs 8-11) ---
        R_full = build_redfield_relaxation_tensor(S_eig, W_mat)
        trace_preservation_err = float(np.linalg.norm(np.einsum("aacd->cd", R_full)))
        assert R_full.shape == (2 * n_v, 2 * n_v, 2 * n_v, 2 * n_v) and trace_preservation_err < 1e-12
        passed += 1
        print(f"[PASS] TODO 8 : build_redfield_relaxation_tensor (||sum_a R_aacd||_F = {trace_preservation_err:.2e})")

        R_sec = apply_secular_approximation(R_full, omega_mat, sec_tol=1e-6)
        assert np.count_nonzero(np.abs(R_sec) > 1e-12) < np.count_nonzero(np.abs(R_full) > 1e-12)
        assert np.linalg.norm(np.einsum("aacd->cd", R_sec)) < 1e-12
        passed += 1
        print("[PASS] TODO 9 : apply_secular_approximation")

        K_pauli, gamma_deph = extract_pauli_rates_and_dephasing(R_sec)
        assert np.linalg.norm(np.sum(K_pauli, axis=0)) < 1e-12 and np.all(gamma_deph >= -1e-15)
        passed += 1
        print(f"[PASS] TODO 10: extract_pauli_rates_and_dephasing (v=1 -> v=0 cooling rate = {K_pauli[n_v, n_v+1]:.5f})")

        L_br_sec = build_redfield_superoperator(omega_mat, R_sec)
        drho_1 = -1j * omega_mat * rho_eig_0 + np.einsum("abcd,cd->ab", R_sec, rho_eig_0)
        drho_2 = (L_br_sec @ rho_eig_0.ravel()).reshape(2 * n_v, 2 * n_v)
        assert np.linalg.norm(drho_1 - drho_2) < 1e-13
        passed += 1
        print("[PASS] TODO 11: build_redfield_superoperator")

        # --- PART 4 CHECKS: VIBRATIONAL COOLING DYNAMICS, T1 & BOLTZMANN EQUILIBRIUM (TODOs 12-15) ---
        rho_dt = rk4_step_redfield(rho_eig_0, omega_mat, R_sec, dt=0.1)
        rho_dt_expm = (expm(L_br_sec * 0.1) @ rho_eig_0.ravel()).reshape(2 * n_v, 2 * n_v)
        assert np.linalg.norm(rho_dt - rho_dt_expm) < 1e-6
        passed += 1
        print("[PASS] TODO 12: rk4_step_redfield")

        traj_sec = propagate_redfield_ver_trajectory(
            rho_eig_0, U_eig, omega_mat, R_sec, Q_vib, N_vib_e, dt=0.25, n_steps=400
        )
        assert np.max(traj_sec["trace_error"]) < 1e-12 and np.min(traj_sec["min_eigenvalue"]) >= -1e-10
        assert traj_sec["mean_n_vib_e"][-1] < 0.25 * traj_sec["mean_n_vib_e"][0]
        passed += 1
        print(f"[PASS] TODO 13: propagate_redfield_ver_trajectory (<N_vib^e> cooled from {traj_sec['mean_n_vib_e'][0]:.3f} -> {traj_sec['mean_n_vib_e'][-1]:.4f})")

        tau_ver, k_ver = fit_vibrational_relaxation_lifetime(
            traj_sec["times"], traj_sec["mean_n_vib_e"], traj_sec["mean_n_vib_e"][-1]
        )
        assert 2.0 < tau_ver < 60.0 and np.isclose(k_ver, 1.0 / tau_ver)
        passed += 1
        print(f"[PASS] TODO 14: fit_vibrational_relaxation_lifetime (VER lifetime T_1 = {tau_ver:.2f}, rate = {k_ver:.4f})")

        boltz_pops, boltz_err = verify_excited_state_boltzmann_equilibrium(
            traj_sec["eigen_populations"][-1], E_evals, n_vib=n_v, temperature_kt=kT_val
        )
        assert boltz_err < 5e-3 and boltz_pops[n_v] > 0.90
        passed += 1
        print(f"[PASS] TODO 15: verify_excited_state_boltzmann_equilibrium (S_1 v=0 Kasha population = {boltz_pops[n_v]:.4f}, Boltzmann error = {boltz_err:.2e})")
        print(f"\nCONGRATULATIONS! All {passed}/{total_todos} Section 07c (Bloch-Redfield Vibrational Relaxation) checkpoints passed!")

    except NotImplementedError as exc:
        print(f"\n[PROGRESS: {passed}/{total_todos} checkpoints passed] Next up -> {exc}")
    except AssertionError as exc:
        print(f"\n[ASSERTION FAILED after {passed}/{total_todos} checkpoints] -> {exc}")


# Uncomment and run as you implement each TODO:
# run_section07c_redfield_checkpoints()
print("Section 07c Redfield checkpoint suite loaded. Implement TODOs 1-15 and call `run_section07c_redfield_checkpoints()`!")

Section 07c Redfield checkpoint suite loaded. Implement TODOs 1-15 and call `run_section07c_redfield_checkpoints()`!
